# Second 1.0 — Phase 6: GRPO Reinforcement Learning on 2B

This notebook trains and evaluates the **Second-1.0-2B-GRPO** model on Google Colab with an **NVIDIA L4 GPU**.
It follows the Phase 6 specification from `PLAN.md`.

### Phase 6 Specification

| Setting | Value |
|---|---|
| **Start From** | Phase 5 SFT LoRA checkpoint (`/content/drive/MyDrive/second/checkpoints/sft-1.0-2b/final_adapter`) |
| **Base Model** | `Qwen/Qwen3.5-2B` (Instruct, text-only, non-thinking) |
| **Algorithm** | GRPO (`TRL` `GRPOTrainer`) |
| **Prompts** | Phase 2 RL prompt set (`data/processed/rl_prompts.jsonl`, 20,000 Lichess prompts) |
| **Generations per Prompt** | 8 |
| **Max Answer Length** | 300 tokens |
| **Temperature** | 0.9 |
| **Learning Rate** | 5e-6 |
| **KL Beta** | 0.02 (0.0 fallback if learning is too slow) |
| **Steps** | 300–600 (default 500, with early stopping when validation reward plateaus) |
| **Checkpoint Strategy** | Save to Google Drive every 50 steps |
| **Reward** | Phase 3 rule checks (`rewards.api.grpo_reward_fn` via `MonitoredGRPOReward`) |
| **GPU** | NVIDIA L4 (or A100) |

### Step-by-Step Instructions

1. **Step 1: Verify GPU Hardware**: Check that an NVIDIA L4 (or A100) GPU is active.
2. **Step 2: Mount Google Drive**: Mount persistent storage and verify the Phase 5 SFT checkpoint.
3. **Step 3: Load Phase 6 Data and Code Bundle**: Extract `phase6_colab_bundle.zip`.
4. **Step 4: Install Dependencies**: Install pinned packages and configure Triton logprob kernels.
5. **Step 5: Load Base Qwen 3.5 2B and Phase 5 SFT LoRA Adapter**: Prepare policy and reference adapters.
6. **Step 6: Prepare RL Prompt Dataset and Monitored Rule Reward**: Load RL prompts and verify reward scoring.
7. **Step 7: Run 50-Step Smoke Check**: Run steps 1–50, verify reward growth, and inspect 20 outputs.
8. **Step 8: Continue GRPO Training to 300–600 Steps**: Train with validation reward tracking and early stopping.
9. **Step 9: Evaluate Top Checkpoints and Generate Frozen Test Set Outputs**: Rank checkpoints and keep the best 2–3.
10. **Step 10: Run Phase 6 Evaluation, Optional Gemini Judge, and Export Results**: Verify Phase 6 rules and download `phase6_grpo_results.zip`.


In [ ]:
# Step 1: Verify GPU Hardware
!nvidia-smi

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise SystemError("No GPU detected. In Colab menu: Runtime -> Change runtime type -> L4 GPU.")

device_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
bf16_supported = torch.cuda.is_bf16_supported()

print(f"GPU Name: {device_name}")
print(f"Total VRAM: {vram_gb:.2f} GB")
print(f"bfloat16 supported: {bf16_supported}")

if "L4" not in device_name and "A100" not in device_name:
    print(f"Notice: Phase 6 targets an NVIDIA L4 or A100 GPU. Detected {device_name}.")


In [ ]:
# Step 2: Mount Google Drive and Locate Phase 5 SFT Adapter
import glob
import os
from pathlib import Path

SFT_CHECKPOINT_ROOT = "/content/drive/MyDrive/second/checkpoints/sft-1.0-2b"
GRPO_CHECKPOINT_DIR = "/content/drive/MyDrive/second/checkpoints/grpo-1.0-2b"
DATA_DRIVE_DIR = "/content/drive/MyDrive/second/data"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(GRPO_CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(DATA_DRIVE_DIR, exist_ok=True)
    print("Google Drive mounted successfully.")
except Exception as exc:
    print(f"Drive mount notice: {exc}")
    SFT_CHECKPOINT_ROOT = "/content/second_checkpoints/sft-1.0-2b"
    GRPO_CHECKPOINT_DIR = "/content/second_checkpoints/grpo-1.0-2b"
    DATA_DRIVE_DIR = "/content/second_data"
    os.makedirs(GRPO_CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(DATA_DRIVE_DIR, exist_ok=True)
    print(f"Using local fallback directory: {GRPO_CHECKPOINT_DIR}")

# Resolve Phase 5 SFT adapter path
sft_candidates = [
    os.path.join(SFT_CHECKPOINT_ROOT, "best_adapter"),
    os.path.join(SFT_CHECKPOINT_ROOT, "final_adapter"),
    SFT_CHECKPOINT_ROOT,
]
sft_step_ckpts = sorted(
    glob.glob(os.path.join(SFT_CHECKPOINT_ROOT, "checkpoint-*")),
    key=lambda x: int(x.split("-")[-1]) if x.split("-")[-1].isdigit() else 0,
)
if sft_step_ckpts:
    sft_candidates.append(sft_step_ckpts[-1])

SFT_ADAPTER_PATH = None
for cand in sft_candidates:
    if os.path.isfile(os.path.join(cand, "adapter_config.json")):
        SFT_ADAPTER_PATH = cand
        break
    if os.path.isfile(os.path.join(cand, "default", "adapter_config.json")):
        SFT_ADAPTER_PATH = os.path.join(cand, "default")
        break

if SFT_ADAPTER_PATH:
    print(f"Found Phase 5 SFT LoRA adapter: {SFT_ADAPTER_PATH}")
else:
    print(
        "WARNING: Phase 5 SFT adapter not found on Drive at "
        f"{SFT_CHECKPOINT_ROOT}. Check that Google Drive has the Phase 5 checkpoint."
    )
print(f"GRPO checkpoint output directory: {GRPO_CHECKPOINT_DIR}")

In [ ]:
# Step 3: Load Phase 6 Data and Code Bundle
import os
import shutil
import sys
import zipfile
from pathlib import Path

WORK_DIR = Path("/content/second_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

bundle_locations = [
    Path("/content/drive/MyDrive/second/data/phase6_colab_bundle.zip"),
    Path("/content/phase6_colab_bundle.zip"),
    Path("/content/data/processed/phase6_colab_bundle.zip"),
]

bundle_path = None
for loc in bundle_locations:
    if loc.is_file():
        bundle_path = loc
        break

if bundle_path is None:
    if Path("eval/testset-1.0.jsonl").is_file() and Path("data/processed/rl_prompts.jsonl").is_file():
        print("Detected local repository checkout. Creating Phase 6 bundle...")
        from training.build_colab_bundle import build_phase6_colab_bundle
        bundle_path = build_phase6_colab_bundle()
    else:
        print("phase6_colab_bundle.zip not found on Drive or /content.")
        print("Please upload phase6_colab_bundle.zip using the file picker:")
        try:
            from google.colab import files
            uploaded = files.upload()
            for filename in uploaded.keys():
                if filename.endswith(".zip"):
                    uploaded_path = Path("/content") / filename
                    drive_copy = Path("/content/drive/MyDrive/second/data") / filename
                    if drive_copy.parent.is_dir():
                        shutil.copyfile(uploaded_path, drive_copy)
                        print(f"Saved copy to Google Drive: {drive_copy}")
                    bundle_path = uploaded_path
                    break
        except Exception as exc:
            print(f"Colab upload notice: {exc}")

if bundle_path and bundle_path.is_file():
    print(f"Extracting bundle from {bundle_path} into {WORK_DIR}...")
    with zipfile.ZipFile(bundle_path, "r") as zf:
        zf.extractall(WORK_DIR)
    print("Bundle extracted successfully.")
else:
    raise FileNotFoundError(
        "Could not find or upload phase6_colab_bundle.zip. "
        "Run python training/build_colab_bundle.py --phase 6 locally and upload the zip."
    )

if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
    print(f"Added {WORK_DIR} to sys.path.")

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")


In [ ]:
# Step 4: Install Dependencies and Configure TRL LogProb Kernels
!pip install -q --upgrade pip
!pip uninstall -y -q torchao torchvision 2>/dev/null || true
!pip install -q "transformers>=4.49.0" "trl>=0.15.0" "accelerate>=1.2.0" "peft>=0.14.0" "datasets>=3.0.0" "bitsandbytes>=0.45.0" "triton" "chess" "pyyaml" "langdetect" "requests"

# Optional fast attention kernels (prebuilt wheels only to avoid compilation delay)
print("Installing optional fast kernels (prebuilt binaries only)...")
!pip install -q causal-conv1d --only-binary :all: 2>/dev/null || echo "causal-conv1d prebuilt binary not found. PyTorch fallback active."
!pip install -q flash-linear-attention --only-binary :all: 2>/dev/null || echo "flash-linear-attention prebuilt binary not found. PyTorch fallback active."

# Ensure Triton ChunkedLogProbFunction is registered in TRL
import importlib
import transformers.utils as tu_trans
tu_trans._triton_available = True
try:
    import trl.kernels.chunked_logprob as clp
    import trl.trainer.utils as tu
    importlib.reload(clp)
    if hasattr(clp, "ChunkedLogProbFunction"):
        tu._ChunkedLogProbFunction = clp.ChunkedLogProbFunction
except Exception as exc:
    print(f"TRL kernel notice: {exc}")

print("Dependencies installed successfully.")


In [ ]:
# Step 5: Load Base Qwen 3.5 2B and Phase 5 SFT LoRA Adapter
import gc
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, PeftModel, get_peft_model, TaskType
import peft.import_utils
import peft.tuners.lora.torchao
from training.grpo_data import load_grpo_config
from training.sft_data import load_sft_config, resolve_text_linear_modules

sft_cfg = load_sft_config("training/configs/second-1.0-2b.yaml")
grpo_cfg = load_grpo_config("training/configs/second-1.0-2b.yaml")

# Bypass incompatible preinstalled torchao version in Google Colab
peft.import_utils.is_torchao_available = lambda: False
peft.tuners.lora.torchao.is_torchao_available = lambda: False

MODEL_ID = grpo_cfg.base_model
tokenizer_source = SFT_ADAPTER_PATH if (SFT_ADAPTER_PATH and os.path.isdir(SFT_ADAPTER_PATH)) else MODEL_ID
print(f"Loading tokenizer from {tokenizer_source}...")
tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

def resolve_adapter_directory(target_path: str | None) -> str | None:
    """Resolve adapter directory containing adapter_config.json."""
    if not target_path or not os.path.isdir(target_path):
        return None
    if os.path.isfile(os.path.join(target_path, "adapter_config.json")):
        return target_path
    if os.path.isfile(os.path.join(target_path, "default", "adapter_config.json")):
        return os.path.join(target_path, "default")
    return None

def load_grpo_policy_model(resume_adapter_path=None):
    """Load base Qwen 3.5 2B and attach trainable Phase 5 SFT LoRA adapter."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    print(f"Loading base model {MODEL_ID} in {'bfloat16' if use_bf16 else 'float32'}...")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16 if use_bf16 else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )

    candidate_path = resume_adapter_path or SFT_ADAPTER_PATH
    adapter_dir = resolve_adapter_directory(candidate_path)
    if adapter_dir:
        print(f"Loading trainable LoRA adapter from: {adapter_dir}")
        # Load single adapter directly so checkpoint saving and resuming operate without nested subdirectories
        peft_model = PeftModel.from_pretrained(
            base_model,
            adapter_dir,
            is_trainable=True,
        )
    else:
        print("WARNING: SFT adapter not found. Initializing fresh LoRA adapter as fallback.")
        target_modules = resolve_text_linear_modules(base_model.named_modules())
        lora_config = LoraConfig(
            r=sft_cfg.lora_r,
            lora_alpha=sft_cfg.lora_alpha,
            lora_dropout=sft_cfg.lora_dropout,
            target_modules=target_modules,
            bias="none",
            task_type=TaskType.CAUSAL_LM,
        )
        peft_model = get_peft_model(base_model, lora_config)

    peft_model.print_trainable_parameters()
    return peft_model

model = load_grpo_policy_model()

In [ ]:
# Step 6: Prepare RL Prompt Dataset and Monitored Rule Reward
from datasets import Dataset
from eval.baselines import load_jsonl
from training.grpo_data import (
    MonitoredGRPOReward,
    load_grpo_config,
    load_rl_prompt_records,
)

grpo_cfg = load_grpo_config("training/configs/second-1.0-2b.yaml")

rl_records = load_rl_prompt_records(
    grpo_cfg.rl_prompts_path,
    max_prompts=20000,
    seed=42,
    shuffle=True,
)
val_items = load_jsonl(grpo_cfg.val_data_path)
test_items = load_jsonl(grpo_cfg.testset_path)

print(f"Loaded {len(rl_records)} RL prompts from {grpo_cfg.rl_prompts_path}.")
print(f"Loaded {len(val_items)} validation items from {grpo_cfg.val_data_path}.")
print(f"Loaded {len(test_items)} frozen test items from {grpo_cfg.testset_path}.")

# Build Hugging Face Dataset with 'prompt' column for GRPOTrainer
rl_dataset = Dataset.from_list([{"prompt": r["prompt"], "id": r["id"]} for r in rl_records])

# Initialize MonitoredGRPOReward (logs every 50 steps and samples 20 outputs)
reward_monitor = MonitoredGRPOReward(log_interval_batches=50, sample_size=20, seed=42)

# Sanity-check the reward function on sample 0 before starting RL
sample_prompt = rl_records[0]["prompt"]
dummy_good = "White plays a solid move in the middlegame to improve piece activity and protect the king."
dummy_bad = "### Heading\n- Bullet list with +1.5 eval centipawn."
test_scores = reward_monitor([sample_prompt, sample_prompt], [dummy_good, dummy_bad])
reward_monitor.call_count = 0
reward_monitor.current_window.clear()

print(f"Reward sanity check -> valid prose score: {test_scores[0]:.4f}, bad format score: {test_scores[1]:.4f}")
assert test_scores[1] == -1.0, "Hard check failure must return -1.0"


In [ ]:
# Step 7: Define Greedy Evaluation Helper and Run 50-Step GRPO Smoke Check
import glob
import inspect
import json
import os
import time
import torch
from trl import GRPOConfig, GRPOTrainer
from eval.metrics import clean_completion_text, evaluate_completions
from training.sft_data import build_non_thinking_chat_prompt

def run_batched_generation(
    records: list[dict],
    eval_model,
    eval_tokenizer,
    batch_size: int = 16,
    max_new_tokens: int = 300,
    model_label: str = "Second-1.0-2B-GRPO",
) -> list[dict]:
    """Generate greedy completions for prompt records with batched inference."""
    orig_padding_side = eval_tokenizer.padding_side
    eval_tokenizer.padding_side = "left"
    if eval_tokenizer.pad_token is None:
        eval_tokenizer.pad_token = eval_tokenizer.eos_token

    stop_ids = []
    if eval_tokenizer.eos_token_id is not None:
        stop_ids.append(int(eval_tokenizer.eos_token_id))
    im_end_id = eval_tokenizer.convert_tokens_to_ids("<|im_end|>")
    if isinstance(im_end_id, int) and im_end_id >= 0 and im_end_id not in stop_ids:
        stop_ids.append(im_end_id)

    target_device = getattr(eval_model, "device", None) or next(eval_model.parameters()).device

    was_training = eval_model.training
    eval_model.eval()

    # Ensure KV-caching is active for fast batched generation
    orig_use_cache = getattr(eval_model.config, "use_cache", True)
    eval_model.config.use_cache = True
    was_grad_ckpt = getattr(eval_model, "is_gradient_checkpointing", False)
    if was_grad_ckpt and hasattr(eval_model, "gradient_checkpointing_disable"):
        eval_model.gradient_checkpointing_disable()

    results = []

    for i in range(0, len(records), batch_size):
        batch = records[i:i + batch_size]
        prompts = [
            r.get("chat_prompt") or build_non_thinking_chat_prompt(r["user_message"])
            for r in batch
        ]
        inputs = eval_tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        ).to(target_device)
        inputs.pop("token_type_ids", None)
        input_len = inputs["input_ids"].shape[1]

        with torch.inference_mode():
            outputs = eval_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
                eos_token_id=stop_ids if stop_ids else eval_tokenizer.eos_token_id,
                pad_token_id=eval_tokenizer.pad_token_id,
            )

        for j, rec in enumerate(batch):
            gen_tokens = outputs[j][input_len:]
            decoded = eval_tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
            cleaned = clean_completion_text(decoded)

            out_rec = {k: v for k, v in rec.items() if k != "comment"}
            out_rec["completion"] = cleaned
            out_rec["output"] = cleaned
            out_rec["model"] = model_label
            results.append(out_rec)

    # Restore model state
    eval_model.config.use_cache = orig_use_cache
    if was_grad_ckpt and hasattr(eval_model, "gradient_checkpointing_enable"):
        eval_model.gradient_checkpointing_enable()
    eval_tokenizer.padding_side = orig_padding_side
    if was_training:
        eval_model.train()
    return results

def build_grpo_config(max_steps: int, beta_val: float, save_steps: int = 50) -> GRPOConfig:
    """Build GRPOConfig compatible with the installed TRL version."""
    is_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    cfg_kwargs = {
        "output_dir": GRPO_CHECKPOINT_DIR,
        "max_steps": max_steps,
        "per_device_train_batch_size": grpo_cfg.per_device_train_batch_size,
        "gradient_accumulation_steps": grpo_cfg.gradient_accumulation_steps,
        "num_generations": grpo_cfg.num_generations,
        "max_completion_length": grpo_cfg.max_completion_length,
        "temperature": grpo_cfg.temperature,
        "learning_rate": grpo_cfg.learning_rate,
        "beta": beta_val,
        "warmup_steps": min(grpo_cfg.warmup_steps, max(1, max_steps // 10)),
        "logging_steps": grpo_cfg.logging_steps,
        "save_strategy": "steps",
        "save_steps": save_steps,
        "save_total_limit": 6,
        "bf16": is_bf16,
        "fp16": torch.cuda.is_available() and not is_bf16,
        "use_vllm": False,
        "report_to": "none",
    }
    sig = inspect.signature(GRPOConfig.__init__)
    if "max_prompt_length" in sig.parameters:
        cfg_kwargs["max_prompt_length"] = grpo_cfg.max_prompt_length
    return GRPOConfig(**cfg_kwargs)

def create_grpo_trainer(policy_model, grpo_args, callbacks=None):
    """Create GRPOTrainer compatible with the installed TRL version."""
    trainer_kwargs = {
        "model": policy_model,
        "args": grpo_args,
        "train_dataset": rl_dataset,
    }
    sig = inspect.signature(GRPOTrainer.__init__)
    if "reward_funcs" in sig.parameters:
        trainer_kwargs["reward_funcs"] = [reward_monitor]
    elif "reward_function" in sig.parameters:
        trainer_kwargs["reward_function"] = reward_monitor
    else:
        trainer_kwargs["reward_funcs"] = [reward_monitor]

    if callbacks:
        trainer_kwargs["callbacks"] = callbacks

    if "processing_class" in sig.parameters:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in sig.parameters:
        trainer_kwargs["tokenizer"] = tokenizer
    return GRPOTrainer(**trainer_kwargs)

# Check existing GRPO checkpoints on Drive
existing_grpo_ckpts = sorted(
    glob.glob(os.path.join(GRPO_CHECKPOINT_DIR, "checkpoint-*")),
    key=lambda x: int(x.split("-")[-1]) if x.split("-")[-1].isdigit() else 0,
)
latest_grpo_ckpt = existing_grpo_ckpts[-1] if existing_grpo_ckpts else None
latest_step = int(latest_grpo_ckpt.split("-")[-1]) if latest_grpo_ckpt else 0

ACTIVE_BETA = grpo_cfg.beta
smoke_start_time = time.time()

if latest_step < 50:
    print("Running Phase 6 Step 1: 50-step GRPO smoke check...")
    smoke_args = build_grpo_config(max_steps=50, beta_val=ACTIVE_BETA, save_steps=50)
    grpo_trainer = create_grpo_trainer(model, smoke_args)
    grpo_trainer.train(resume_from_checkpoint=latest_grpo_ckpt)
    smoke_duration = time.time() - smoke_start_time
    print(f"50-step smoke check completed in {smoke_duration / 60.0:.2f} minutes.")

    if reward_monitor.step_summaries:
        s50 = reward_monitor.step_summaries[-1]
        print("\n--- Step 50 Window Diagnostics ---")
        print(f"Mean Reward:      {s50['mean_reward']:.4f}")
        print(f"Hard-Fail Rate:   {s50['hard_fail_rate']:.1f}%")
        print(f"Avg Word Count:   {s50['avg_word_count']:.1f} words")
        print(f"Length Collapse:  {s50['length_collapse']}")
        print(f"Robotic Openings: {s50['robotic_text']} (top: '{s50['top_opening_prefix']}', {s50['top_opening_share']*100:.1f}%)")

    sampled_50 = reward_monitor.sampled_outputs.get(50, [])
    if not sampled_50 and reward_monitor.sampled_outputs:
        sampled_50 = reward_monitor.sampled_outputs[max(reward_monitor.sampled_outputs.keys())]
    print(f"\n--- Inspecting {len(sampled_50)} Sampled Outputs at Step 50 (Section 3.4) ---")
    for idx, sample in enumerate(sampled_50[:20], start=1):
        print(f"[{idx:02d}] reward={sample['reward']:.4f} | words={sample['word_count']} | hard_pass={sample['passed_all_hard']} | text={sample['completion'][:140]}")
else:
    print(f"Found existing checkpoint at step {latest_step} (>= 50). Skipping 50-step smoke run.")

In [ ]:
# Step 8: Continue GRPO Training (Up to 500 Steps) with Validation Tracking and Early Stopping
import json
import os
import time
from transformers import TrainerCallback

TARGET_MAX_STEPS = grpo_cfg.max_steps  # Default: 500 (in 300-600 range)
SAVE_INTERVAL = grpo_cfg.save_steps    # Default: 50
EARLY_STOP_PATIENCE = 3                # Stop after 3 eval intervals (150 steps) without val reward improvement after step 300

checkpoint_val_outputs_map = {}
checkpoint_val_metrics_map = {}
step_progression_logs = list(reward_monitor.step_summaries)

# Load any previously saved step progression logs from Drive
step_logs_file = os.path.join(GRPO_CHECKPOINT_DIR, "grpo_step_logs.json")
if os.path.isfile(step_logs_file):
    try:
        with open(step_logs_file, "r", encoding="utf-8") as f:
            saved_logs = json.load(f)
            if isinstance(saved_logs, list) and len(saved_logs) > len(step_progression_logs):
                step_progression_logs = saved_logs
    except Exception as exc:
        print(f"Could not read previous step logs: {exc}")

class Phase6ValidationCallback(TrainerCallback):
    """Log reward metrics, evaluate validation split every 50 steps, and apply early stopping."""

    def __init__(self, eval_model=None, eval_tokenizer=None, min_steps: int = 300, patience: int = 3):
        self.eval_model = eval_model
        self.eval_tokenizer = eval_tokenizer
        self.min_steps = min_steps
        self.patience = patience
        self.best_val_reward = -999.0
        self.no_improve_count = 0

    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        ckpt_name = f"checkpoint-{step}"
        active_model = kwargs.get("model") or self.eval_model or model
        active_tokenizer = kwargs.get("processing_class") or self.eval_tokenizer or tokenizer

        print(f"\n[Step {step}] Running validation split evaluation for {ckpt_name}...")
        val_preds = run_batched_generation(
            val_items,
            active_model,
            active_tokenizer,
            batch_size=16,
            max_new_tokens=grpo_cfg.max_completion_length,
            model_label=f"Second-1.0-2B-GRPO-{ckpt_name}",
        )
        val_eval = evaluate_completions(val_preds)
        v_agg = val_eval.aggregate

        # Save checkpoint validation outputs to Drive and local results
        os.makedirs("eval/results", exist_ok=True)
        local_val_file = f"eval/results/{ckpt_name}_val_outputs.jsonl"
        drive_val_file = os.path.join(GRPO_CHECKPOINT_DIR, f"{ckpt_name}_val_outputs.jsonl")
        for out_p in (local_val_file, drive_val_file):
            with open(out_p, "w", encoding="utf-8") as f:
                for r in val_preds:
                    f.write(json.dumps(r) + "\n")

        checkpoint_val_outputs_map[ckpt_name] = local_val_file
        checkpoint_val_metrics_map[ckpt_name] = val_eval

        log_entry = {
            "step": step,
            "checkpoint": ckpt_name,
            "mean_reward": float(v_agg.mean_reward),
            "hard_fail_rate": float(v_agg.hard_fail_rate),
            "avg_word_count": float(v_agg.avg_word_count_overall),
            "pct_in_band": float(v_agg.pct_in_band_overall),
        }
        step_progression_logs.append(log_entry)
        with open(step_logs_file, "w", encoding="utf-8") as f:
            json.dump(step_progression_logs, f, indent=2)

        print(
            f"[Step {step}] Val Mean Reward: {v_agg.mean_reward:.4f} | "
            f"Val Hard-Fail: {v_agg.hard_fail_rate:.1f}% | "
            f"Val Avg Words: {v_agg.avg_word_count_overall:.1f}"
        )

        # Print 5 sample validation outputs at each 50-step checkpoint
        for s_idx, item in enumerate(val_eval.items[:5], start=1):
            print(f"  Val Sample {s_idx}: reward={item.reward:.4f} | words={item.word_count} | {item.completion[:120]}")

        if v_agg.mean_reward > self.best_val_reward + 1e-4:
            self.best_val_reward = v_agg.mean_reward
            self.no_improve_count = 0
        else:
            self.no_improve_count += 1

        if step >= self.min_steps and self.no_improve_count >= self.patience:
            print(
                f"[Early Stopping] Validation reward did not improve for {self.patience} "
                f"checks after step {self.min_steps}. Stopping at step {step}."
            )
            control.should_training_stop = True
        return control

# Re-scan checkpoints before continuing
existing_grpo_ckpts = sorted(
    glob.glob(os.path.join(GRPO_CHECKPOINT_DIR, "checkpoint-*")),
    key=lambda x: int(x.split("-")[-1]) if x.split("-")[-1].isdigit() else 0,
)
latest_grpo_ckpt = existing_grpo_ckpts[-1] if existing_grpo_ckpts else None
latest_step = int(latest_grpo_ckpt.split("-")[-1]) if latest_grpo_ckpt else 0

# If checkpoint exists, ensure active model weights match latest checkpoint
if latest_grpo_ckpt and latest_step > 0:
    print(f"Ensuring policy model is aligned with latest checkpoint at step {latest_step}...")
    model = load_grpo_policy_model(resume_adapter_path=latest_grpo_ckpt)

val_callback = Phase6ValidationCallback(
    eval_model=model,
    eval_tokenizer=tokenizer,
    min_steps=300,
    patience=EARLY_STOP_PATIENCE,
)

# Evaluate step-50 checkpoint if it exists and was not evaluated yet
if latest_grpo_ckpt and f"checkpoint-{latest_step}" not in checkpoint_val_metrics_map:
    class _DummyState:
        global_step = latest_step
    class _DummyControl:
        should_training_stop = False
    val_callback.on_save(None, _DummyState(), _DummyControl(), model=model)

train_start_time = time.time()
if latest_step < TARGET_MAX_STEPS:
    print(f"Continuing GRPO training from step {latest_step} to {TARGET_MAX_STEPS}...")
    full_args = build_grpo_config(max_steps=TARGET_MAX_STEPS, beta_val=ACTIVE_BETA, save_steps=SAVE_INTERVAL)
    grpo_trainer = create_grpo_trainer(model, full_args, callbacks=[val_callback])
    train_result = grpo_trainer.train(resume_from_checkpoint=latest_grpo_ckpt)
    total_train_minutes = round((time.time() - train_start_time) / 60.0, 2)
    final_step = grpo_trainer.state.global_step
    final_loss = float(getattr(train_result, "training_loss", 0.0))
else:
    print(f"GRPO training already reached step {latest_step}. Proceeding to checkpoint ranking.")
    total_train_minutes = 0.0
    final_step = latest_step
    final_loss = 0.0

# Save final adapter
final_adapter_dir = os.path.join(GRPO_CHECKPOINT_DIR, "final_adapter")
model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)
print(f"Saved final GRPO adapter to: {final_adapter_dir}")

In [ ]:
# Step 9: Rank Checkpoints, Prune Old Checkpoints (Keep Best 2-3), and Generate Frozen Test Set Outputs
import glob
import shutil
from eval.metrics import evaluate_file
from training.evaluate_grpo import rank_checkpoints

os.makedirs("eval/results", exist_ok=True)

# Discover all saved checkpoint validation outputs on Drive and locally
for val_f in sorted(glob.glob(os.path.join(GRPO_CHECKPOINT_DIR, "checkpoint-*_val_outputs.jsonl"))):
    cname = os.path.basename(val_f).split("_val_outputs")[0]
    local_copy = os.path.join("eval/results", os.path.basename(val_f))
    if not os.path.exists(local_copy):
        shutil.copyfile(val_f, local_copy)
    checkpoint_val_outputs_map[cname] = local_copy
    if cname not in checkpoint_val_metrics_map:
        checkpoint_val_metrics_map[cname] = evaluate_file(local_copy)

# If any checkpoint directory on Drive lacks validation outputs, evaluate it now
all_ckpt_dirs = sorted(
    glob.glob(os.path.join(GRPO_CHECKPOINT_DIR, "checkpoint-*")),
    key=lambda x: int(x.split("-")[-1]) if x.split("-")[-1].isdigit() else 0,
)
for ckpt_dir in all_ckpt_dirs:
    if not os.path.isdir(ckpt_dir):
        continue
    cname = os.path.basename(ckpt_dir)
    if cname not in checkpoint_val_metrics_map:
        print(f"Evaluating checkpoint {cname} on validation split...")
        model = load_grpo_policy_model(resume_adapter_path=ckpt_dir)
        val_preds = run_batched_generation(
            val_items,
            model,
            tokenizer,
            batch_size=16,
            max_new_tokens=grpo_cfg.max_completion_length,
            model_label=f"Second-1.0-2B-GRPO-{cname}",
        )
        local_val_file = f"eval/results/{cname}_val_outputs.jsonl"
        drive_val_file = os.path.join(GRPO_CHECKPOINT_DIR, f"{cname}_val_outputs.jsonl")
        for out_p in (local_val_file, drive_val_file):
            with open(out_p, "w", encoding="utf-8") as f:
                for r in val_preds:
                    f.write(json.dumps(r) + "\n")
        checkpoint_val_outputs_map[cname] = local_val_file
        checkpoint_val_metrics_map[cname] = evaluate_file(local_val_file)

# Rank checkpoints by validation reward and keep the best 3 checkpoints
rankings, top_3_checkpoints = rank_checkpoints(checkpoint_val_metrics_map, top_k=3)
best_ckpt_name = top_3_checkpoints[0] if top_3_checkpoints else "final_adapter"
print(f"\nTop checkpoints by validation reward: {top_3_checkpoints}")
print(f"Selected best checkpoint: {best_ckpt_name}")

# Remove non-top checkpoint directories on Drive to save storage (keep top 3 + final_adapter)
for ckpt_dir in all_ckpt_dirs:
    cname = os.path.basename(ckpt_dir)
    if os.path.isdir(ckpt_dir) and cname not in top_3_checkpoints:
        print(f"Pruning lower-ranked checkpoint directory: {cname}")
        shutil.rmtree(ckpt_dir, ignore_errors=True)

# Load the winning checkpoint adapter and generate frozen test set outputs
best_ckpt_path = os.path.join(GRPO_CHECKPOINT_DIR, best_ckpt_name)
if os.path.isdir(best_ckpt_path):
    model = load_grpo_policy_model(resume_adapter_path=best_ckpt_path)
    # Copy winning adapter to best_adapter directory on Drive
    best_adapter_dir = os.path.join(GRPO_CHECKPOINT_DIR, "best_adapter")
    model.save_pretrained(best_adapter_dir)
    tokenizer.save_pretrained(best_adapter_dir)
    print(f"Saved winning checkpoint ({best_ckpt_name}) to: {best_adapter_dir}")

# Copy winning validation outputs to canonical grpo-1.0-2b_val_outputs.jsonl
if best_ckpt_name in checkpoint_val_outputs_map:
    best_val_src = checkpoint_val_outputs_map[best_ckpt_name]
    shutil.copyfile(best_val_src, "eval/results/grpo-1.0-2b_val_outputs.jsonl")
    shutil.copyfile(best_val_src, os.path.join(GRPO_CHECKPOINT_DIR, "grpo-1.0-2b_val_outputs.jsonl"))
else:
    val_outputs = run_batched_generation(
        val_items,
        model,
        tokenizer,
        batch_size=16,
        max_new_tokens=grpo_cfg.max_completion_length,
        model_label="Second-1.0-2B-GRPO",
    )
    for p in ("eval/results/grpo-1.0-2b_val_outputs.jsonl", os.path.join(GRPO_CHECKPOINT_DIR, "grpo-1.0-2b_val_outputs.jsonl")):
        with open(p, "w", encoding="utf-8") as f:
            for r in val_outputs:
                f.write(json.dumps(r) + "\n")

print(f"Generating greedy outputs for {len(test_items)} frozen test set items using {best_ckpt_name}...")
test_outputs = run_batched_generation(
    test_items,
    model,
    tokenizer,
    batch_size=16,
    max_new_tokens=grpo_cfg.max_completion_length,
    model_label="Second-1.0-2B-GRPO",
)

for p in ("eval/results/grpo-1.0-2b_outputs.jsonl", os.path.join(GRPO_CHECKPOINT_DIR, "grpo-1.0-2b_outputs.jsonl")):
    with open(p, "w", encoding="utf-8") as f:
        for r in test_outputs:
            f.write(json.dumps(r) + "\n")
    print(f"Saved frozen test set outputs: {p}")

effective_final_step = globals().get("final_step", latest_step if "latest_step" in globals() else 0)
effective_final_loss = globals().get("final_loss", 0.0)
effective_train_minutes = globals().get("total_train_minutes", 0.0)
effective_beta = globals().get("ACTIVE_BETA", grpo_cfg.beta)

training_summary = {
    "model_id": MODEL_ID,
    "sft_adapter_path": SFT_ADAPTER_PATH,
    "best_checkpoint": best_ckpt_name,
    "top_3_checkpoints": top_3_checkpoints,
    "global_steps": effective_final_step,
    "train_loss": effective_final_loss,
    "beta": effective_beta,
    "learning_rate": grpo_cfg.learning_rate,
    "num_generations": grpo_cfg.num_generations,
    "temperature": grpo_cfg.temperature,
    "training_time_minutes": effective_train_minutes,
    "step_logs": step_progression_logs,
}
summary_path = os.path.join(GRPO_CHECKPOINT_DIR, "grpo_training_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(training_summary, f, indent=2)
shutil.copyfile(summary_path, "eval/results/grpo_training_summary.json")
print(f"Saved GRPO training summary to {summary_path}")

In [ ]:
# Step 10: Run Phase 6 Evaluation, Optional Gemini Judge, and Export Results Bundle
import os
import shutil
import zipfile
from pathlib import Path
from training.evaluate_grpo import run_phase6_evaluation

# Check if GEMINI_API_KEY is stored in Colab Secrets
if not os.environ.get("GEMINI_API_KEY"):
    try:
        from google.colab import userdata
        secret_key = userdata.get("GEMINI_API_KEY")
        if secret_key:
            os.environ["GEMINI_API_KEY"] = secret_key
            print("Loaded GEMINI_API_KEY from Colab Secrets.")
    except Exception:
        print("GEMINI_API_KEY not set in Colab Secrets. Judge can be run locally after downloading results.")

print("Executing Phase 6 evaluation runner...")
summary = run_phase6_evaluation(
    val_outputs_path="eval/results/grpo-1.0-2b_val_outputs.jsonl",
    test_outputs_path="eval/results/grpo-1.0-2b_outputs.jsonl",
    sft_outputs_path="eval/results/sft-1.0-2b_outputs.jsonl",
    base_qwen_path="eval/results/baselines/base-qwen-2b.jsonl",
    results_dir="eval/results",
    reports_dir="eval/reports",
    model_name="grpo-1.0-2b",
    best_checkpoint=best_ckpt_name,
    checkpoint_val_paths=checkpoint_val_outputs_map if checkpoint_val_outputs_map else None,
    step_logs=step_progression_logs,
    training_metadata=training_summary,
    run_judge=True,
    skip_judge_if_no_key=True,
)

print("\n" + "=" * 60)
print("PHASE 6 RELEASE RULE VERIFICATION:")
print(f"- Overall Pass Status: {'PASS' if summary['passed_phase6'] else 'FAIL'}")
print(f"- Best Checkpoint: {summary['best_checkpoint']}")
print(f"- Format Pass Rate: {summary['pass_rule']['grpo_format_pass_rate']:.1f}% (Required: > 95.0%)")
print(f"- GRPO Mean Reward: {summary['pass_rule']['grpo_mean_reward']:.4f} (SFT: {summary['pass_rule']['sft_mean_reward']:.4f})")
print(f"- GRPO Hard-Fail Rate: {summary['pass_rule']['grpo_hard_fail_rate']:.1f}% (SFT: {summary['pass_rule']['sft_hard_fail_rate']:.1f}%, Base: {summary['pass_rule']['base_hard_fail_rate']:.1f}%)")
print("=" * 60 + "\n")

# Package all outputs, checkpoint validation files, and reports into phase6_grpo_results.zip
results_zip_name = "phase6_grpo_results.zip"
files_to_pack = [
    ("eval/results/grpo-1.0-2b_val_outputs.jsonl", "grpo-1.0-2b_val_outputs.jsonl"),
    ("eval/results/grpo-1.0-2b_outputs.jsonl", "grpo-1.0-2b_outputs.jsonl"),
    ("eval/results/grpo-1.0-2b_val_eval.json", "grpo-1.0-2b_val_eval.json"),
    ("eval/results/grpo-1.0-2b_eval.json", "grpo-1.0-2b_eval.json"),
    ("eval/results/grpo_training_summary.json", "grpo_training_summary.json"),
    ("eval/reports/grpo-1.0-2b.md", "grpo-1.0-2b.md"),
]
if os.path.exists("eval/results/grpo-1.0-2b_judge.json"):
    files_to_pack.append(("eval/results/grpo-1.0-2b_judge.json", "grpo-1.0-2b_judge.json"))
for cname, cpath in checkpoint_val_outputs_map.items():
    if os.path.exists(cpath):
        files_to_pack.append((cpath, os.path.basename(cpath)))

seen_arcs = set()
with zipfile.ZipFile(results_zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for src, arc in files_to_pack:
        if os.path.exists(src) and arc not in seen_arcs:
            zf.write(src, arcname=arc)
            seen_arcs.add(arc)

drive_zip_path = os.path.join(GRPO_CHECKPOINT_DIR, results_zip_name)
shutil.copyfile(results_zip_name, drive_zip_path)
print(f"Saved results zip to Google Drive: {drive_zip_path}")

try:
    from google.colab import files
    files.download(results_zip_name)
    print(f"Triggered browser download for {results_zip_name}.")
except Exception as exc:
    print(f"Browser download notice: {exc}")

# Stop the runtime when the run ends to save Colab compute units
try:
    from google.colab import runtime
    print("Stopping Colab runtime to save compute units...")
    runtime.unassign()
except Exception as exc:
    print(f"Runtime disconnect notice: {exc}")